In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import re

def auto_clean_uci_dataset(dataset_id, naming_strategy='description', save_path=None):
    """
    Automatically fetch and clean any UCI dataset
    
    Parameters:
    - dataset_id: UCI dataset ID number
    - naming_strategy: 'description' | 'original' | 'both'
        - 'description': Use variable descriptions as column names
        - 'original': Keep original names but standardize format
        - 'both': Prefix original name with shortened description
    - save_path: Where to save CSV (optional)
    """
    
    # Fetch data
    data = fetch_ucirepo(id=dataset_id)
    df = data.data.original  # Includes ID column
    var_info = data.variables
    
    print(f"Dataset: {data.metadata.get('name', 'Unknown')}")
    print(f"Original shape: {df.shape}")
    print(f"Original columns: {len(df.columns)}")
    print("="*80)
    
    def clean_text(text, max_length=40):
        """Clean and standardize text for column name"""
        text = str(text).strip()
        # Remove content in parentheses
        text = re.sub(r'\([^)]*\)', '', text)
        # Take first part before colon or semicolon
        text = text.split(':')[0].split(';')[0].split(',')[0]
        # Remove special characters except spaces
        text = re.sub(r'[^\w\s]', '', text)
        # Truncate if too long
        if len(text) > max_length:
            text = text[:max_length]
        # Standardize: uppercase with underscores
        text = text.strip().upper().replace(' ', '_')
        text = re.sub(r'_+', '_', text).strip('_')
        return text
    
    column_mapping = {}
    
    for idx, row in var_info.iterrows():
        old_name = row['name']
        
        if naming_strategy == 'description' and 'description' in row and pd.notna(row['description']):
            new_name = clean_text(row['description'])
        elif naming_strategy == 'both' and 'description' in row and pd.notna(row['description']):
            desc = clean_text(row['description'], max_length=20)
            orig = clean_text(old_name, max_length=15)
            new_name = f"{desc}_{orig}" if desc != orig else orig
        else:
            new_name = clean_text(old_name)
        
        # Ensure unique names
        base_name = new_name
        counter = 1
        while new_name in column_mapping.values():
            new_name = f"{base_name}_{counter}"
            counter += 1
        
        column_mapping[old_name] = new_name
    
    # Apply renaming
    df_clean = df.rename(columns=column_mapping)
    
    # Print mapping
    print("\nColumn Renaming:")
    for old, new in column_mapping.items():
        print(f"  {old:25s} -> {new}")
    
    print(f"\nCleaned columns: {df_clean.columns.tolist()}")
    print(f"Final shape: {df_clean.shape}")
    
    # Save if requested
    if save_path:
        df_clean.to_csv(save_path, index=False)
        print(f"\n✓ Saved to: {save_path}")
    
    return df_clean, column_mapping

# Example usage with different datasets:

# Credit Card Default Dataset
df1, mapping1 = auto_clean_uci_dataset(
    dataset_id=350,
    naming_strategy='description',
    save_path='../../data/default_of_credit_card_clients.csv'
)

# Try with any other UCI dataset - it will work automatically!
# df2 = auto_clean_uci_dataset(dataset_id=19)  # Heart Disease
# df3 = auto_clean_uci_dataset(dataset_id=53)  # Iris

Dataset: Default of Credit Card Clients
Original shape: (30000, 25)
Original columns: 25

Column Renaming:
  ID                        -> ID
  X1                        -> LIMIT_BAL
  X2                        -> SEX
  X3                        -> EDUCATION
  X4                        -> MARRIAGE
  X5                        -> AGE
  X6                        -> PAY_0
  X7                        -> PAY_2
  X8                        -> PAY_3
  X9                        -> PAY_4
  X10                       -> PAY_5
  X11                       -> PAY_6
  X12                       -> BILL_AMT1
  X13                       -> BILL_AMT2
  X14                       -> BILL_AMT3
  X15                       -> BILL_AMT4
  X16                       -> BILL_AMT5
  X17                       -> BILL_AMT6
  X18                       -> PAY_AMT1
  X19                       -> PAY_AMT2
  X20                       -> PAY_AMT3
  X21                       -> PAY_AMT4
  X22                       -> PAY_AMT5